# Logistic regression baseline: validation only

Experimental output. Not a clinical decision.

Run with the repository root as the kernel working directory and the pinned dependencies installed. All workflow code is imported from `src/`. This notebook freezes patient-grouped chronological split IDs, fits preprocessing and a prespecified logistic regression on training requests, and evaluates **validation only**. It does not transform test features, produce test predictions, compute test performance, or tune on the test set.

## Split design

Cutoffs are selected from request timestamps using nominal training/validation/test windows of 70%/15%/15%, without consulting labels. Each patient belongs to the window of their first labelled request. Requests from that patient in later windows are excluded, rather than moved across dates. Requests tied at a cutoff go entirely into the later window. Consequently, retained validation and test fractions can be smaller than the nominal windows. All retained training dates precede validation dates, which precede test dates; patient IDs never overlap between retained partitions. The partition targets later requests from newly encountered labelled-patient cohorts.

`artifacts/splits.json` saves request IDs, cutoff dates, the exclusion rule, source fingerprint, and split summaries. `artifacts/split_class_counts.csv` gives class counts, patients, request counts, and date ranges, including excluded requests. Test labels are used only for this requested descriptive class-count table, never for model selection or evaluation. Strict chronology is feasible here, so no group-fold fallback is used. Infeasible chronological inputs fail explicitly.

Earlier EDA and hypothesis testing used the full cohort. This is therefore a newly reserved modelling holdout, not a fully blinded holdout relative to that earlier exploration. No prior exploration is presented as test performance.

## Preprocessing and baseline specification

The unfitted [ColumnTransformer](https://scikit-learn.org/1.6/modules/generated/sklearn.compose.ColumnTransformer.html) comes from `src/feature_engineering.py` and is fitted only inside the training pipeline:

- Numeric intake features: training-only median imputation followed by scaling. Entirely missing training columns retain their position with a documented zero fallback. Duration uses log-transformed hours, not raw seconds or redundant duration conversions.
- Severity and pain: preserve their fixed ordinal codes; encode missing or unseen codes as a distinct negative value. These schema codes do not learn the urgency target.
- Categorical intake fields: fixed missing token, then one-hot encoding learned on training, with unknown categories ignored at inference.
- Multi-hot items, explicit unknown/none responses, and vital-missingness indicators: pass through from deterministic intake engineering. Unknown multi-value counts remain missing until the training-only numeric imputer.
- Optional TF-IDF: disabled for this baseline. Set `INCLUDE_TEXT=True` to include submission-time `symptom_description` from raw intake for training/validation requests only. Missing descriptions use a fixed token; vocabulary and IDF learn only on training. Assessment notes are never used.

Identifiers, raw timestamps, birth dates, target labels, and non-target assessment fields cannot enter the feature allowlist. The classifier uses fixed regularisation and balanced class weights derived only from training labels, with the repository random seed. There is no parameter search, threshold tuning, resampling, or refitting on validation. Future model selection must prioritise High/Critical recall and under-triage, then macro F1, calibration, and stability.


In [1]:
from src.train import show_baseline_results, train_baseline

INCLUDE_TEXT = False
model, validation_metrics = train_baseline(include_text=INCLUDE_TEXT)


     split  requests  patients  fraction_of_all_requests               start                 end  Low  Moderate  High  Critical
     train     10366      7664                  0.699980 2025-07-01T01:21:23 2026-04-10T21:38:11 3712      3319  2391       944
validation      1179      1114                  0.079614 2026-04-10T23:16:32 2026-05-21T17:58:38  428       384   273        94
      test      1074      1014                  0.072523 2026-05-21T17:59:04 2026-06-30T22:43:20  380       380   228        86
  excluded      2190      1836                  0.147883 2026-04-10T21:46:25 2026-06-30T23:39:23  690       730   535       235
Validation metrics only:
{
  "high_recall": 0.5860805860805861,
  "critical_recall": 0.8191489361702128,
  "macro_f1": 0.6888289318252737,
  "accuracy": 0.7065309584393554,
  "balanced_accuracy": 0.7153674467620766
}
{
  "n": 1179,
  "correct_count": 833,
  "under_triage_count": 162,
  "under_triage_rate": 0.13740458015267176,
  "over_triage_count": 184,
  "

## Saved validation results

The tables below read the generated artifact files. The confusion matrix uses true labels as rows and predictions as columns, ordered Low, Moderate, High, Critical. Class-specific precision, recall, F1, and support accompany accuracy, balanced accuracy, macro F1, and weighted F1. Metrics are unweighted even though fitting uses balanced class weights.

Under-triage means the predicted ordinal rank is lower than the true rank; over-triage means it is higher. Severe under-triage is exactly Critical→Low/Moderate or High→Low. Overall rates divide by all validation requests. Per-true-class rates divide by that class's support; an absent class has a null rate. The additional severe rate among High/Critical cases uses only those true classes as its denominator.

`artifacts/baseline_metrics.json` contains validation metrics only. `artifacts/baseline_validation_predictions.csv` contains only validation request IDs, true labels, and predictions. `artifacts/baseline_training.json` records fit provenance and feature names. The complete fitted pipeline is saved in `models/baseline_logistic_regression.joblib`.


In [2]:
show_baseline_results()


,split,requests,patients,fraction_of_all_requests,start,end,Low,Moderate,High,Critical
0,train,10366,7664,0.699980,2025-07-01T01:21:23,2026-04-10T21:38:11,3712,3319,2391,944
1,validation,1179,1114,0.079614,2026-04-10T23:16:32,2026-05-21T17:58:38,428,384,273,94
2,test,1074,1014,0.072523,2026-05-21T17:59:04,2026-06-30T22:43:20,380,380,228,86
3,excluded,2190,1836,0.147883,2026-04-10T21:46:25,2026-06-30T23:39:23,690,730,535,235


**Validation only. Experimental output. Not a clinical decision.**

,high_recall,critical_recall,macro_f1,weighted_f1,accuracy,balanced_accuracy
0,0.586081,0.819149,0.688829,0.704029,0.706531,0.715367


,precision,recall,f1-score,support
Low,0.786813,0.836449,0.810872,428.0
Moderate,0.670423,0.619792,0.644114,384.0
High,0.689655,0.586081,0.633663,273.0
Critical,0.562044,0.819149,0.666667,94.0
macro avg,0.677234,0.715367,0.688829,1179.0
weighted avg,0.708487,0.706531,0.704029,1179.0


,Low,Moderate,High,Critical
Low,358,65,5,0
Moderate,90,238,54,2
High,7,48,160,58
Critical,0,4,13,77


,under_triage_count,under_triage_rate,over_triage_count,over_triage_rate,severe_under_triage_count,severe_under_triage_rate
0,162,0.137405,184,0.156064,11,0.00933


,requests
Critical_to_Low,0
Critical_to_Moderate,4
High_to_Low,7


The full labelled cohort was explored in earlier EDA/hypothesis analyses. This is a newly reserved modelling holdout, not fully blinded to prior exploration. No baseline model transform, prediction, evaluation, or tuning uses test requests.

## Interpretation and next evaluation stage

Inspect High and Critical recall and both under-triage measures before judging aggregate accuracy. These descriptive validation results are experimental decision-support outputs, not clinical decisions or evidence of clinical safety. No override or escalation rule is applied to predictions to improve these metrics. A future dashboard must independently escalate red-flag responses regardless of model output.

Use validation for later development choices while keeping the saved test IDs reserved until final evaluation. The newly reserved holdout's earlier full-cohort exploration remains a limitation. This notebook makes no test-performance claim.
